# Classic Chunking And One-Vector Baselines

Start from the situation many teams already have:

- one vector for the whole document
- or one vector for each fixed-size chunk

This notebook asks the practical question:

- what did that simplification make easier?
- what kind of miss did it create?

We keep the token vectors fixed so the thing that moves is the retrieval geometry, not the encoder.

Backstage verification note: the deterministic bridges in this notebook are covered by `python.tests.test_course_chunking_baselines_smoke`.

In [ ]:
from pathlib import Path
import json
import sys

import numpy as np


def find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "python" / "kayak").exists():
            return candidate
    raise RuntimeError("Run this notebook from the repository or one of its subdirectories.")


REPO_ROOT = find_repo_root(Path.cwd().resolve())
sys.path.insert(0, str(REPO_ROOT / "python"))

import kayak
from kayak_bridge.judged_metrics import summarize_ranked_task

CACHE_ROOT = REPO_ROOT / ".cache" / "kayak"
print("Working from repo root:", REPO_ROOT)
print("Backends available here:", kayak.available_backends())

In [ ]:
def mean_vec(matrix: np.ndarray) -> np.ndarray:
    return np.mean(matrix, axis=0, dtype=np.float32, keepdims=True).astype(np.float32)


def build_onevec_doc_index(task: dict) -> kayak.LateIndex:
    return kayak.documents(
        [row["doc_id"] for row in task["documents"]],
        [mean_vec(np.asarray(row["vectors"], dtype=np.float32)) for row in task["documents"]],
        texts=[row["text"] for row in task["documents"]],
    ).pack()


def build_chunked_index(task: dict, chunk_size: int) -> tuple[kayak.LateIndex, dict[str, str]]:
    chunk_ids = []
    chunk_vectors = []
    parent_by_chunk: dict[str, str] = {}
    for row in task["documents"]:
        matrix = np.asarray(row["vectors"], dtype=np.float32)
        for chunk_index, start in enumerate(range(0, len(matrix), chunk_size)):
            chunk = matrix[start : start + chunk_size]
            chunk_id = f"{row['doc_id']}::chunk{chunk_index}"
            chunk_ids.append(chunk_id)
            chunk_vectors.append(mean_vec(chunk))
            parent_by_chunk[chunk_id] = row["doc_id"]
    return kayak.documents(chunk_ids, chunk_vectors).pack(), parent_by_chunk


def dedup_parent_docs(chunk_hits, parent_by_chunk: dict[str, str], k: int) -> tuple[str, ...]:
    ranked_doc_ids = []
    seen = set()
    for hit in chunk_hits:
        doc_id = parent_by_chunk[hit.doc_id]
        if doc_id in seen:
            continue
        seen.add(doc_id)
        ranked_doc_ids.append(doc_id)
        if len(ranked_doc_ids) >= k:
            break
    return tuple(ranked_doc_ids)


## Toy: Chunking Can Help When The Evidence Is Local

Here the relevant document really does have the answer, but it is buried inside one small local region.
If you compress the whole document to one vector, that local signal gets washed out.
If you chunk it, that little evidence pocket gets a chance to stand on its own.


In [ ]:
DIM = 64
TOKEN_TO_INDEX: dict[str, int] = {}


def token_vector(token: str) -> np.ndarray:
    index = TOKEN_TO_INDEX.setdefault(token, len(TOKEN_TO_INDEX))
    vector = np.zeros(DIM, dtype=np.float32)
    vector[index] = np.float32(1.0)
    return vector


def encode_tokens(tokens: list[str]) -> np.ndarray:
    return np.stack([token_vector(token) for token in tokens])


docs = {
    "relevant": ["noise1", "noise2", "noise3", "noise4", "cancel", "subscription", "noise5", "noise6"],
    "partial": ["cancel", "noise7", "cancel", "noise8"],
    "other": ["invoice", "billing"],
}
query_tokens = ["cancel", "subscription"]

onevec_index = kayak.documents(
    list(docs),
    [mean_vec(encode_tokens(tokens)) for tokens in docs.values()],
).pack()
query_onevec = kayak.query(mean_vec(encode_tokens(query_tokens)))

chunk_ids = []
chunk_vectors = []
parent = {}
for doc_id, tokens in docs.items():
    matrix = encode_tokens(tokens)
    for chunk_index, start in enumerate(range(0, len(matrix), 2)):
        chunk_id = f"{doc_id}::chunk{chunk_index}"
        chunk_ids.append(chunk_id)
        chunk_vectors.append(mean_vec(matrix[start : start + 2]))
        parent[chunk_id] = doc_id

chunk_index = kayak.documents(chunk_ids, chunk_vectors).pack()

print("If I keep one vector per document:", [(hit.doc_id, hit.score) for hit in kayak.search(query_onevec, onevec_index, k=3)])
chunk_hits = kayak.search(query_onevec, chunk_index, k=chunk_index.document_count)
print("If I chunk first and then dedupe docs:", dedup_parent_docs(chunk_hits, parent, 3))

## Toy: Chunking Can Hurt When The Evidence Spans Chunks

Now the answer only becomes obvious when you keep several concepts together.
Exact late interaction on the full document can still see that whole conjunction.
Chunked one-vector retrieval breaks the evidence apart and loses the shape that mattered.


In [ ]:
docs = {
    "relevant": ["alpha", "noise1", "beta", "noise2", "gamma", "noise3"],
    "partial": ["alpha", "beta", "noise4", "noise5"],
    "other": ["invoice", "billing"],
}
query_tokens = ["alpha", "beta", "gamma"]

exact_index = kayak.documents(
    list(docs),
    [encode_tokens(tokens) for tokens in docs.values()],
).pack()
exact_query = kayak.query(encode_tokens(query_tokens))
print("If I keep the full document together:", [(hit.doc_id, hit.score) for hit in kayak.search(exact_query, exact_index, k=3)])

query_onevec = kayak.query(mean_vec(encode_tokens(query_tokens)))
chunk_ids = []
chunk_vectors = []
parent = {}
for doc_id, tokens in docs.items():
    matrix = encode_tokens(tokens)
    for chunk_index, start in enumerate(range(0, len(matrix), 2)):
        chunk_id = f"{doc_id}::chunk{chunk_index}"
        chunk_ids.append(chunk_id)
        chunk_vectors.append(mean_vec(matrix[start : start + 2]))
        parent[chunk_id] = doc_id

chunk_index = kayak.documents(chunk_ids, chunk_vectors).pack()
chunk_hits = kayak.search(query_onevec, chunk_index, k=chunk_index.document_count)
print("If I chunk first and then dedupe docs:", dedup_parent_docs(chunk_hits, parent, 3))

## Real Judged-Slice Summary

Now leave the toy and look at judged slices that are already cached locally.
The comparison below asks the same real-life question on labeled data:

- if I keep exact late interaction as the reference path,
- how much quality do I lose when I collapse to one vector,
- and when does chunking help or hurt instead?


In [ ]:
TASK_PATHS = [
    CACHE_ROOT / "limit_small_real_subset" / "python_task.json",
    CACHE_ROOT / "bright_stackoverflow_real_subset" / "python_task.json",
    CACHE_ROOT / "legal_rag_bench_real_subset" / "python_task.json",
    CACHE_ROOT / "r2med_biology_real_subset" / "python_task.json",
]

summary_rows = []
for path in TASK_PATHS:
    task = json.loads(path.read_text())
    exact_index = kayak.documents(
        [row["doc_id"] for row in task["documents"]],
        [np.asarray(row["vectors"], dtype=np.float32) for row in task["documents"]],
        texts=[row["text"] for row in task["documents"]],
    ).pack()
    onevec_index = build_onevec_doc_index(task)
    chunk16_index, parent16 = build_chunked_index(task, 16)
    chunk32_index, parent32 = build_chunked_index(task, 32)

    exact_ranked = []
    onevec_ranked = []
    chunk16_ranked = []
    chunk32_ranked = []

    for row in task["queries"]:
        query_matrix = np.asarray(row["vectors"], dtype=np.float32)
        exact_query = kayak.query(query_matrix, text=row["text"])
        onevec_query = kayak.query(mean_vec(query_matrix), text=row["text"])

        exact_hits = kayak.search(exact_query, exact_index, k=task["k"], backend=kayak.NUMPY_REFERENCE_BACKEND)
        onevec_hits = kayak.search(onevec_query, onevec_index, k=task["k"], backend=kayak.NUMPY_REFERENCE_BACKEND)
        chunk16_hits = kayak.search(onevec_query, chunk16_index, k=chunk16_index.document_count, backend=kayak.NUMPY_REFERENCE_BACKEND)
        chunk32_hits = kayak.search(onevec_query, chunk32_index, k=chunk32_index.document_count, backend=kayak.NUMPY_REFERENCE_BACKEND)

        exact_ranked.append(tuple(hit.doc_id for hit in exact_hits))
        onevec_ranked.append(tuple(hit.doc_id for hit in onevec_hits))
        chunk16_ranked.append(dedup_parent_docs(chunk16_hits, parent16, task["k"]))
        chunk32_ranked.append(dedup_parent_docs(chunk32_hits, parent32, task["k"]))

    exact_summary = summarize_ranked_task(task=task, ranked_doc_ids_by_query=exact_ranked)
    onevec_summary = summarize_ranked_task(task=task, ranked_doc_ids_by_query=onevec_ranked)
    chunk16_summary = summarize_ranked_task(task=task, ranked_doc_ids_by_query=chunk16_ranked)
    chunk32_summary = summarize_ranked_task(task=task, ranked_doc_ids_by_query=chunk32_ranked)

    summary_rows.append(
        {
            "slice_name": task["slice_name"],
            "primary_metric": task["primary_metric"],
            "exact": round(exact_summary.primary_value, 4),
            "onevec": round(onevec_summary.primary_value, 4),
            "chunk16": round(chunk16_summary.primary_value, 4),
            "chunk32": round(chunk32_summary.primary_value, 4),
            "exact_recall": round(exact_summary.mean_recall_at_k, 4),
            "onevec_recall": round(onevec_summary.mean_recall_at_k, 4),
            "chunk16_recall": round(chunk16_summary.mean_recall_at_k, 4),
            "chunk32_recall": round(chunk32_summary.mean_recall_at_k, 4),
        }
    )

summary_rows

## Practical Takeaway

If your current system already uses one vector per document or one vector per chunk, this notebook is not here to shame that choice.
It is here to help you ask a better next question:

- did chunking isolate a useful local signal?
- or did chunking just create a different failure?
- how much judged quality is the simplification already costing me?

That is the real-life reason to keep exact late interaction around as a diagnosis tool.
